# 時系列予測

**学習目標**
 1. Keras で線形、DNN、および CNN モデルを構築して株式市場の動向を予測する。
 2. Keras でシンプルな RNN モデルとマルチレイヤー RNN モデルを構築する。
 3. RNN と CNN のアーキテクチャを組み合わせて、株式市場の動向を予測する Keras モデルを作成する。
 
このラボでは、前のラボで作成した株式市場データセットを使用して、株式市場の動向を予測するカスタム Keras モデルを構築します。まずは線形モデル、DNN、および CNN モデルから始めます。

このモデルの特徴量は本質的にシーケンシャル（時系列）であるため、次に Keras でさまざまな RNN モデルを構築する方法を説明します。シンプルな RNN モデルから始めて、Keras でマルチレイヤー RNN を作成する方法を確認します。また、1 次元 CNN の特徴量と典型的な RNN アーキテクチャを組み合わせる方法も説明します。

このノートブックでは、さまざまなモデルタイプを探索します。結果を追跡するために、検証セットに対する精度をここの表に記録してください。機械学習において「万能（one-size-fits-all）」なものは滅多にないため、各モデルでさまざまなハイパーパラメータ（トレーニングステップ、正則化、学習率、オプティマイザ、バッチサイズなど）を自由にテストしてください。以下の表でモデルのパフォーマンスを記録していきましょう。

|  モデル   | 検証精度（Validation Accuracy）  |
|----------|:---------------:|
| Baseline |    0.295         |
| Linear   |    --        |
| DNN      |    --         |
| 1-d CNN  |    --         |
| simple RNN  |    --         |
| multi-layer RNN  |    --         |
| RNN using CNN features  |    --         |
| CNN using RNN features  |    --         |

## 必要なライブラリの読み込みと環境変数の設定

In [ ]:
import os

os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from google.cloud import bigquery
from keras.layers import (
    GRU,
    LSTM,
    RNN,
    Bidirectional,
    Conv1D,
    Dense,
    Flatten,
    MaxPool1D,
    Reshape,
)
from keras.models import Sequential
from keras.optimizers import Adam
from keras.regularizers import L1

# きれいな図を描画するため
%matplotlib inline
mpl.rc("axes", labelsize=14)
mpl.rc("xtick", labelsize=12)
mpl.rc("ytick", labelsize=12)

# 再現性のある結果のため。
from numpy.random import seed

seed(1)

In [ ]:
PROJECT = !(gcloud config get-value core/project)
PROJECT = PROJECT[0]
%env PROJECT = {PROJECT}
%env BUCKET = {PROJECT}
%env REGION = "us-central1"

## 時系列データの探索

まずは BigQuery から時系列データの小さなサンプルを抽出し、モデリングに向けてデータをクリーニングするためのヘルパー関数を作成します。BigQuery の `percent_change_sp500` テーブルのデータを使用します。`close_values_prior_260` 列には、特定の銘柄に関する過去 260 日間の終値が含まれています。

In [ ]:
%%time
bq = bigquery.Client(project=PROJECT)

bq_query = """
#standardSQL
SELECT
  symbol,
  Date,
  direction,
  close_values_prior_260
FROM
  `stock_market.eps_percent_change_sp500`
LIMIT
  100
"""

df_stock_raw = bq.query(bq_query).to_dataframe()

In [ ]:
df_stock_raw.head()

以下の `clean_data` 関数は 3 つのことを行います。
 1. まず、inf または NA 値を削除します。
 2. 次に、`Date` フィールドをパースして文字列として読み込みます。
 3. 最後に、ラベル `direction` を数値に変換し、'DOWN' を 0、'STAY' を 1、'UP' を 2 にマッピングします。

In [ ]:
def clean_data(input_df):
    """Cleans data to prepare for training.

    Args:
        input_df: Pandas dataframe.
    Returns:
        Pandas dataframe.
    """
    df = input_df.copy()

    # TF は DataFrame 内の datetime を受け付けません。
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df["Date"] = df["Date"].dt.strftime("%Y-%m-%d")

    # TF は数値ラベルを必要とします。
    df["direction_numeric"] = df["direction"].apply(
        lambda x: {"DOWN": 0, "STAY": 1, "UP": 2}[x]
    )
    return df

In [ ]:
df_stock = clean_data(df_stock_raw)

In [ ]:
df_stock.head()

## データの読み込みと前処理

モデリングを開始する前に、z スコア（標準化）にスケーリングして特徴量を前処理します。これにより、モデルに供給される特徴量の値の範囲が同等になり、勾配降下中の収束に役立ちます。

In [ ]:
STOCK_HISTORY_COLUMN = "close_values_prior_260"
COL_NAMES = ["day_" + str(day) for day in range(0, 260)]
LABEL = "direction_numeric"

In [ ]:
def _scale_features(df):
    """z-scale feature columns of Pandas dataframe.

    Args:
        features: Pandas dataframe.
    Returns:
        Pandas dataframe with each column standardized according to the
        values in that column.
    """
    avg = df.mean()
    std = df.std()
    return (df - avg) / std


def create_features(df, label_name):
    """Create modeling features and label from Pandas dataframe.

    Args:
        df: Pandas dataframe.
        label_name: str, the column name of the label.
    Returns:
        Pandas dataframe
    """
    # 終値のリストを含む 1 つの列を 260 列に展開します。
    time_series_features = df[STOCK_HISTORY_COLUMN].apply(pd.Series)

    # 列名を変更。
    time_series_features.columns = COL_NAMES
    time_series_features = _scale_features(time_series_features)

    # 時系列特徴量を静的特徴量およびラベルと連結します。
    label_column = df[LABEL]

    return pd.concat([time_series_features, label_column], axis=1)

In [ ]:
df_features = create_features(df_stock, LABEL)

In [ ]:
df_features.head()

いくつかの例をプロットして、前処理ステップが正しく実装されていることを確認しましょう。

In [ ]:
ix_to_plot = [0, 1, 9, 5]
fig, ax = plt.subplots(1, 1, figsize=(15, 8))
for ix in ix_to_plot:
    label = df_features["direction_numeric"].iloc[ix]
    example = df_features[COL_NAMES].iloc[ix]
    ax = example.plot(label=label, ax=ax)
    ax.set_ylabel("scaled price")
    ax.set_xlabel("prior days")
    ax.legend()

### トレーニング/評価/テストへの分割の作成

次に、トレーニング/検証/テストデータセットの再現可能な分割を作成し、これらのデータセットをローカルの CSV ファイルに保存します。以下のクエリは、データセット全体のサブサンプルを取得し、トレーニング/検証/テストセット用に 70-15-15 の分割を作成します。

In [ ]:
def _create_split(phase):
    """Create string to produce train/valid/test splits for a SQL query.

    Args:
        phase: str, either TRAIN, VALID, or TEST.
    Returns:
        String.
    """
    floor, ceiling = "2002-11-01", "2010-07-01"
    if phase == "VALID":
        floor, ceiling = "2010-07-01", "2011-09-01"
    elif phase == "TEST":
        floor, ceiling = "2011-09-01", "2012-11-30"
    return """
    WHERE Date >= '{}'
    AND Date < '{}'
    """.format(
        floor, ceiling
    )


def create_query(phase):
    """Create SQL query to create train/valid/test splits on subsample.

    Args:
        phase: str, either TRAIN, VALID, or TEST.
        sample_size: str, amount of data to take for subsample.
    Returns:
        String.
    """
    basequery = """
    #standardSQL
    SELECT
      symbol,
      Date,
      direction,
      close_values_prior_260
    FROM
      `stock_market.eps_percent_change_sp500`
    """

    return basequery + _create_split(phase)

In [ ]:
bq = bigquery.Client(project=PROJECT)

for phase in ["TRAIN", "VALID", "TEST"]:
    # 1. クエリ文字列の作成
    query_string = create_query(phase)
    # 2. 結果を DataFrame にロード
    df = bq.query(query_string).to_dataframe()

    # 3. DataFrame のクリーニングと前処理
    df = clean_data(df)
    df = create_features(df, label_name="direction_numeric")

    # 3. DataFrame を CSV に書き込み
    if not os.path.exists("../data"):
        os.mkdir("../data")
    df.to_csv(
        f"../data/stock-{phase.lower()}.csv",
        index_label=False,
        index=False,
    )
    print(
        "Wrote {} lines to {}".format(
            len(df), f"../data/stock-{phase.lower()}.csv"
        )
    )

In [ ]:
ls -la ../data

## モデリング

実験目的として、上記で作成した `.csv` ファイルを使用して、メモリに収まるデータでさまざまなモデルをトレーニングします。

In [ ]:
N_TIME_STEPS = 260
N_LABELS = 3

Xtrain = pd.read_csv("../data/stock-train.csv")
Xvalid = pd.read_csv("../data/stock-valid.csv")

ytrain = Xtrain.pop(LABEL)
yvalid = Xvalid.pop(LABEL)

ytrain_categorical = keras.utils.to_categorical(ytrain.values)
yvalid_categorical = keras.utils.to_categorical(yvalid.values)

トレーニングの進行状況を監視し、さまざまなモデルの評価指標を比較するために、以下の関数を使用して、トレーニングと検証の損失や精度など、トレーニングジョブから取得された指標をプロットします。

In [ ]:
def plot_curves(train_data, val_data, label="Accuracy"):
    """Plot training and validation metrics on single axis.

    Args:
        train_data: list, metrics obtrained from training data.
        val_data: list, metrics obtained from validation data.
        label: str, title and label for plot.
    Returns:
        Matplotlib plot.
    """
    plt.plot(
        np.arange(len(train_data)) + 0.5,
        train_data,
        "b.-",
        label="Training " + label,
    )
    plt.plot(
        np.arange(len(val_data)) + 1,
        val_data,
        "r.-",
        label="Validation " + label,
    )
    plt.gca().xaxis.set_major_locator(mpl.ticker.MaxNLocator(integer=True))
    plt.legend(fontsize=14)
    plt.xlabel("Epochs")
    plt.ylabel(label)
    plt.grid(True)

### ベースライン

Keras でモデリングを開始する前に、シンプルなヒューリスティックを使用してベンチマークを作成しましょう。トレーニングセットの最頻クラスを常に予測した場合、検証セットでどのような精度が得られるかを確認します。

In [ ]:
sum(yvalid == ytrain.value_counts().idxmax()) / yvalid.shape[0]

最も一般的な結果である `UP` を単純に予測するだけでも、検証セットで約 29.5% の精度が得られます。

### 線形モデル

シンプルな線形モデルから始め、シーケンシャル入力を単一の全結合（Dense）レイヤーにマッピングします。

In [ ]:
# TODO 1a
model = Sequential()
model.add(
    Dense(
        units=N_LABELS,
        activation="softmax",
        kernel_regularizer=L1(0.1),
    )
)

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

history = model.fit(
    x=Xtrain.values,
    y=ytrain_categorical,
    batch_size=1000,
    validation_data=(Xvalid.values, yvalid_categorical),
    epochs=5,
)

In [ ]:
plot_curves(history.history["loss"], history.history["val_loss"], label="Loss")

In [ ]:
plot_curves(
    history.history["accuracy"],
    history.history["val_accuracy"],
    label="Accuracy",
)

精度はかなり早く横ばいになるようです。精度を報告するために、トレーニングの最後の数エポックにわたる検証セットでの精度を平均します。

In [ ]:
np.mean(history.history["val_accuracy"][-5:])

### ディープニューラルネットワーク（DNN）

線形モデルは、ナイーブなベンチマークよりも改善されています。より複雑なモデルを使用すれば、さらに良い結果が得られるかもしれません。次に、Keras を使用してディープニューラルネットワーク（DNN）を作成します。ここでは 2 層の DNN で実験しますが、パフォーマンスを向上させるために、より複雑なモデルを試したり、他の手法を追加したりしても構いません。

In [ ]:
# TODO 1b
dnn_hidden_units = [16, 8]

model = Sequential()
for layer in dnn_hidden_units:
    model.add(Dense(units=layer, activation="relu"))

model.add(
    Dense(
        units=N_LABELS,
        activation="softmax",
        kernel_regularizer=L1(0.1),
    )
)

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

history = model.fit(
    x=Xtrain.values,
    y=ytrain_categorical,
    batch_size=1000,
    validation_data=(Xvalid.values, yvalid_categorical),
    epochs=5,
)

In [ ]:
plot_curves(history.history["loss"], history.history["val_loss"], label="Loss")

In [ ]:
plot_curves(
    history.history["accuracy"],
    history.history["val_accuracy"],
    label="Accuracy",
)

In [ ]:
np.mean(history.history["val_accuracy"][-5:])

### 畳み込みニューラルネットワーク（CNN）

DNN の方がわずかに良い結果を示しました。次に、畳み込みニューラルネットワークのパフォーマンスを見てみましょう。

1 次元畳み込みは、時系列データから特徴量を抽出したり、データセットのより短い固定長のセグメントから特徴量を導出したりするのに役立ちます。[Keras における Conv1d](https://keras.io/api/layers/convolution_layers/convolution1d/) の実装方法についてはドキュメントをご覧ください。マックスプーリングは、畳み込みニューラルネットワークとともによく使用されるダウンサンプリング戦略です。次に、Keras で `Conv1D` を使用して畳み込みレイヤーを作成し、全結合レイヤーに渡す前に `MaxPool1D` を使用してマックスプーリングを実行する CNN モデルを構築します。

In [ ]:
# TODO 1c
model = Sequential()

# 畳み込みレイヤー
model.add(Reshape(target_shape=[N_TIME_STEPS, 1]))
model.add(
    Conv1D(
        filters=5,
        kernel_size=5,
        strides=2,
        padding="valid",
        input_shape=[None, 1],
    )
)
model.add(MaxPool1D(pool_size=2, strides=None, padding="valid"))


# 結果を平坦化して DNN に渡す。
model.add(Flatten())
model.add(Dense(units=N_TIME_STEPS // 4, activation="relu"))

model.add(
    Dense(
        units=N_LABELS,
        activation="softmax",
        kernel_regularizer=L1(0.1),
    )
)

model.compile(
    optimizer=Adam(learning_rate=0.01),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

history = model.fit(
    x=Xtrain.values,
    y=ytrain_categorical,
    batch_size=1000,
    validation_data=(Xvalid.values, yvalid_categorical),
    epochs=5,
)

In [ ]:
plot_curves(history.history["loss"], history.history["val_loss"], label="Loss")

In [ ]:
plot_curves(
    history.history["accuracy"],
    history.history["val_accuracy"],
    label="Accuracy",
)

In [ ]:
np.mean(history.history["val_accuracy"][-5:])

### リカレントニューラルネットワーク（RNN）

RNN は、時系列データの学習に特に適しています。あるセルの出力を次のステップの入力として供給することにより、あるイテレーションから次のイテレーションへと状態情報を保持します。以下のセルでは、Keras で RNN モデルを構築します。RNN の最終状態がキャプチャされ、全結合レイヤーに渡されて予測が生成されます。

In [ ]:
# TODO 2a
model = Sequential()

# RNN レイヤーに渡すために入力をリシェイプ。
model.add(Reshape(target_shape=[N_TIME_STEPS, 1]))
model.add(LSTM(N_TIME_STEPS // 8))

model.add(
    Dense(
        units=N_LABELS,
        activation="softmax",
        kernel_regularizer=L1(0.1),
    )
)

# モデルの作成。
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

history = model.fit(
    x=Xtrain.values,
    y=ytrain_categorical,
    batch_size=1000,
    validation_data=(Xvalid.values, yvalid_categorical),
    epochs=5,
)

In [ ]:
plot_curves(history.history["loss"], history.history["val_loss"], label="Loss")

In [ ]:
plot_curves(
    history.history["accuracy"],
    history.history["val_accuracy"],
    label="Accuracy",
)

In [ ]:
np.mean(history.history["val_accuracy"][-5:])

### マルチレイヤー RNN

次に、マルチレイヤー RNN を構築します。ディープニューラルネットワークの複数のレイヤーによってトレーニング中により複雑な特徴量を学習できるように、追加の RNN レイヤーによって時系列データの複雑な特徴量を学習できる可能性があります。マルチレイヤー RNN では、最初の RNN レイヤーの出力が次の RNN レイヤーへの入力として供給されます。

In [ ]:
# TODO 2b
rnn_hidden_units = [N_TIME_STEPS // 16, N_TIME_STEPS // 32]

model = Sequential()

# RNN レイヤーに渡すために入力をリシェイプ。
model.add(Reshape(target_shape=[N_TIME_STEPS, 1]))

for layer in rnn_hidden_units[:-1]:
    model.add(GRU(units=layer, return_sequences=True))

model.add(GRU(units=rnn_hidden_units[-1]))
model.add(
    Dense(
        units=N_LABELS,
        activation="softmax",
        kernel_regularizer=L1(0.1),
    )
)

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

history = model.fit(
    x=Xtrain.values,
    y=ytrain_categorical,
    batch_size=1000,
    validation_data=(Xvalid.values, yvalid_categorical),
    epochs=5,
)

In [ ]:
plot_curves(history.history["loss"], history.history["val_loss"], label="Loss")

In [ ]:
plot_curves(
    history.history["accuracy"],
    history.history["val_accuracy"],
    label="Accuracy",
)

In [ ]:
np.mean(history.history["val_accuracy"][-5:])

### CNN と RNN のアーキテクチャの組み合わせ

最後に、畳み込みネットワークとリカレントネットワークの両方の側面を組み合わせたモデルアーキテクチャを見ていきます。例えば、1 次元畳み込みレイヤーを使用してシーケンスを処理し、予測の前に RNN モデルに渡される特徴量を作成できます。

In [ ]:
# TODO 3a
model = Sequential()

# 畳み込みレイヤーのために入力をリシェイプ
model.add(Reshape(target_shape=[N_TIME_STEPS, 1]))

model.add(
    Conv1D(
        filters=20,
        kernel_size=4,
        strides=2,
        padding="valid",
        input_shape=[None, 1],
    )
)
model.add(MaxPool1D(pool_size=2, strides=None, padding="valid"))

model.add(
    LSTM(
        units=N_TIME_STEPS // 2,
        return_sequences=False,
        kernel_regularizer=L1(0.1),
    )
)
model.add(Dense(units=N_LABELS, activation="softmax"))

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

history = model.fit(
    x=Xtrain.values,
    y=ytrain_categorical,
    batch_size=1000,
    validation_data=(Xvalid.values, yvalid_categorical),
    epochs=5,
)

In [ ]:
plot_curves(history.history["loss"], history.history["val_loss"], label="Loss")

In [ ]:
plot_curves(
    history.history["accuracy"],
    history.history["val_accuracy"],
    label="Accuracy",
)

In [ ]:
np.mean(history.history["val_accuracy"][-5:])

また、1 次元 CNN を使用して RNN の出力から特徴量を作成するハイブリッドモデルの構築を試すこともできます。

In [ ]:
# TODO 3b
rnn_hidden_units = [N_TIME_STEPS // 32, N_TIME_STEPS // 64]

model = Sequential()

# 入力をリシェイプして RNN レイヤーに渡す。
model.add(Reshape(target_shape=[N_TIME_STEPS, 1]))
for layer in rnn_hidden_units:
    model.add(LSTM(layer, return_sequences=True))

# RNN の出力に 1 次元畳み込みを適用。
model.add(Conv1D(filters=5, kernel_size=3, strides=2, padding="valid"))
model.add(MaxPool1D(pool_size=4, strides=None, padding="valid"))

# 畳み込みの出力を平坦化して DNN に渡す。
model.add(Flatten())
model.add(
    Dense(
        units=N_TIME_STEPS // 32,
        kernel_regularizer=L1(0.1),
    )
)
model.add(
    Dense(
        units=N_LABELS,
        activation="softmax",
        kernel_regularizer=L1(0.1),
    )
)

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

history = model.fit(
    x=Xtrain.values,
    y=ytrain_categorical,
    batch_size=1000,
    validation_data=(Xvalid.values, yvalid_categorical),
    epochs=5,
)

In [ ]:
plot_curves(history.history["loss"], history.history["val_loss"], label="Loss")

In [ ]:
plot_curves(
    history.history["accuracy"],
    history.history["val_accuracy"],
    label="Accuracy",
)

In [ ]:
np.mean(history.history["val_accuracy"][-5:])

## エクストラクレジット（発展課題）

1. `eps_percent_change_sp500` テーブルには、各サンプルの静的特徴量も含まれています。具体的には、過去 5 日間、20 日間、260 日間の `MAX`、`MIN`、`AVG`、`STD` をキャプチャした集計情報を含む、前のラボで作成したエンジニアリングされた特徴量です（例: `close_MIN_prior_5_days`、`close_MIN_prior_20_days`、`close_MIN_prior_260_days` など）。上記で使用したシーケンス特徴量に加えて、これらの特徴量を組み込んだモデルを構築してみてください。パフォーマンスは向上しますか？

2. `eps_percent_change_sp500` テーブルには、一株当たり利益に関する情報をキャプチャする `surprise` 特徴量も含まれています。上記で使用したシーケンス特徴量に加えて、`surprise` 特徴量を使用するモデルを構築してみてください。パフォーマンスは向上しますか？

Copyright 2025 Google Inc. Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License